In [1]:
import pandas as pd

urls = pd.read_csv("openphish.txt", header=None, names=["url"])
urls["label"] = "phishing"

In [2]:
urls

,url,label
0,http://instagram-clone-login.vercel.app/,phishing
1,https://www.robiox.com.gr/users/548531114/profile,phishing
2,http://map-iphone.com/QPP,phishing
3,http://www.tiny.cc/jd94101/,phishing
4,http://thwmigj.yudasise.com/,phishing
...,...,...
295,http://sim-app.info/dMq,phishing
296,https://s4w.in/https-roblox-com-users-22015010...,phishing
297,https://west.cougars.hair/,phishing
298,http://sim-app.info/sUk,phishing


In [5]:
urls.to_csv('openphish.csv', index=False)


In [15]:
import pandas as pd
from urllib.parse import urlparse

# --------------------------------------------------
# Extract domain from URL
# --------------------------------------------------
def extract_domain(url):
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except:
        return None


# ==================================================
# OPENPHISH
# ==================================================
openphish = pd.read_csv("openphish.csv")

openphish["domain"] = openphish["url"].apply(extract_domain)
openphish["label"] = "phishing"

openphish = openphish[["domain", "label"]]


# ==================================================
# PHISHTANK
# ==================================================
phishtank = pd.read_csv("verified_online.csv")

phishtank["domain"] = phishtank["url"].apply(extract_domain)
phishtank["label"] = "phishing"

phishtank = phishtank[["domain", "label"]]


# ==================================================
# TRANCO TOP DOMAINS
# ==================================================
tranco = pd.read_csv(
    "top-1m.csv",
    header=None,
    names=["rank", "domain"]
)

# use top 20k domains
tranco = tranco.head(20000)

tranco["label"] = "legitimate"

tranco = tranco[["domain", "label"]]


# ==================================================
# COMBINE
# ==================================================
combined = pd.concat(
    [openphish, phishtank, tranco],
    ignore_index=True
)

combined.dropna(inplace=True)

combined["domain"] = combined["domain"].str.strip()

combined.drop_duplicates(
    subset=["domain"],
    inplace=True
)

print(combined["label"].value_counts())

combined.to_csv(
    "combined_domains.csv",
    index=False
)

print(f"Saved {len(combined)} domains")

label
phishing      32791
legitimate    19904
Name: count, dtype: int64
Saved 52695 domains


In [16]:
import pandas as pd
import numpy as np
import math
from collections import Counter

df = pd.read_csv("combined_domains.csv")


BRANDS = [
    "google",
    "facebook",
    "amazon",
    "paypal",
    "microsoft",
    "apple",
    "instagram",
    "netflix",
    "bank",
    "sbi",
    "hdfc",
    "icici"
]

KEYWORDS = [
    "login",
    "secure",
    "verify",
    "account",
    "update"
]


def entropy(domain):
    prob = [
        float(domain.count(c)) / len(domain)
        for c in dict.fromkeys(list(domain))
    ]

    return -sum(
        p * math.log2(p)
        for p in prob
    )


def extract_features(domain):

    parts = domain.split(".")

    tld = parts[-1]

    return {
        "length": len(domain),

        "subdomain_count": max(
            len(parts)-2,
            0
        ),

        "hyphen_count":
            domain.count("-"),

        "digit_count":
            sum(c.isdigit() for c in domain),

        "entropy":
            entropy(domain),

        "has_login":
            int("login" in domain),

        "has_secure":
            int("secure" in domain),

        "has_verify":
            int("verify" in domain),

        "has_account":
            int("account" in domain),

        "has_update":
            int("update" in domain),

        "brand_match":
            int(
                any(
                    brand in domain
                    for brand in BRANDS
                )
            ),

        "has_punycode":
            int("xn--" in domain),

        "tld":
            tld
    }


features = df["domain"].apply(extract_features)

feature_df = pd.DataFrame(features.tolist())

final_df = pd.concat(
    [df, feature_df],
    axis=1
)

final_df.to_csv(
    "domain_features.csv",
    index=False
)

print(final_df.head())

                             domain     label  length  subdomain_count  \
0  instagram-clone-login.vercel.app  phishing      32                1   
1                     robiox.com.gr  phishing      13                1   
2                    map-iphone.com  phishing      14                0   
3                           tiny.cc  phishing       7                0   
4              thwmigj.yudasise.com  phishing      20                1   

   hyphen_count  digit_count   entropy  has_login  has_secure  has_verify  \
0             2            0  3.905639          1           0           0   
1             0            0  3.026987          0           0           0   
2             1            0  3.378783          0           0           0   
3             0            0  2.521641          0           0           0   
4             0            0  3.921928          0           0           0   

   has_account  has_update  brand_match  has_punycode  tld  
0            0           0     

In [17]:
df = pd.get_dummies(
    final_df,
    columns=["tld"]
)

In [18]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("domain_features.csv")

X = df.drop(
    ["domain", "label"],
    axis=1
)

X = pd.get_dummies(
    X,
    columns=["tld"]
)

y = df["label"]


X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print(X_train.shape)
print(X_valid.shape)
print(X_test.shape)

(36886, 533)
(7904, 533)
(7905, 533)


In [22]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier


# =====================================================
# LOAD DATA
# =====================================================

df = pd.read_csv("domain_features.csv")

print("Dataset Shape:", df.shape)

# =====================================================
# FEATURES & LABEL
# =====================================================

X = df.drop(
    columns=["domain", "label"]
)

# One-Hot Encode TLD
X = pd.get_dummies(
    X,
    columns=["tld"]
)

y = df["label"]

# =====================================================
# LABEL ENCODING
# =====================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("\nClasses:")
for i, c in enumerate(label_encoder.classes_):
    print(i, "->", c)

# =====================================================
# TRAIN / VALID / TEST
# =====================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.30,
    stratify=y_encoded,
    random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("\nTrain:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

# =====================================================
# XGBOOST MODEL
# =====================================================

model = XGBClassifier(
    objective="binary:logistic",

    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42
)

# =====================================================
# TRAIN
# =====================================================

print("\nTraining XGBoost...")

model.fit(
    X_train,
    y_train
)

print("Training Complete")

# =====================================================
# VALIDATION
# =====================================================

valid_preds = model.predict(X_valid)

print("\n==============================")
print("VALIDATION RESULTS")
print("==============================")

print(
    "Accuracy:",
    accuracy_score(y_valid, valid_preds)
)

print(
    "Precision:",
    precision_score(
        y_valid,
        valid_preds,
        average="weighted"
    )
)

print(
    "Recall:",
    recall_score(
        y_valid,
        valid_preds,
        average="weighted"
    )
)

print(
    "F1 Score:",
    f1_score(
        y_valid,
        valid_preds,
        average="weighted"
    )
)

# =====================================================
# TEST EVALUATION
# =====================================================

test_preds = model.predict(X_test)

print("\n==============================")
print("TEST RESULTS")
print("==============================")

print(
    "Accuracy:",
    accuracy_score(y_test, test_preds)
)

print(
    "Precision:",
    precision_score(
        y_test,
        test_preds,
        average="weighted"
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        test_preds,
        average="weighted"
    )
)

print(
    "F1 Score:",
    f1_score(
        y_test,
        test_preds,
        average="weighted"
    )
)

print("\nClassification Report")

print(
    classification_report(
        y_test,
        test_preds,
        target_names=label_encoder.classes_
    )
)

print("\nConfusion Matrix")

print(
    confusion_matrix(
        y_test,
        test_preds
    )
)

# =====================================================
# FEATURE IMPORTANCE
# =====================================================

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

print("\nTop 20 Features")

print(
    importance_df.head(20)
)

# =====================================================
# SAVE ARTIFACTS
# =====================================================

joblib.dump(
    model,
    "phishing_xgboost.joblib"
)

joblib.dump(
    label_encoder,
    "label_encoder.joblib"
)

joblib.dump(
    list(X.columns),
    "feature_columns.joblib"
)

print("\nSaved:")
print("phishing_xgboost.joblib")
print("label_encoder.joblib")
print("feature_columns.joblib")

Dataset Shape: (52695, 15)

Classes:
0 -> legitimate
1 -> phishing

Train: (36886, 533)
Validation: (7904, 533)
Test: (7905, 533)

Training XGBoost...
Training Complete

VALIDATION RESULTS
Accuracy: 0.9186487854251012
Precision: 0.9250053471392987
Recall: 0.9186487854251012
F1 Score: 0.9194745867363061

TEST RESULTS
Accuracy: 0.9263757115749526
Precision: 0.9306563621263781
Recall: 0.9263757115749526
F1 Score: 0.9270040266832007

Classification Report
              precision    recall  f1-score   support

  legitimate       0.86      0.96      0.91      2986
    phishing       0.97      0.91      0.94      4919

    accuracy                           0.93      7905
   macro avg       0.92      0.93      0.92      7905
weighted avg       0.93      0.93      0.93      7905


Confusion Matrix
[[2853  133]
 [ 449 4470]]

Top 20 Features
             feature  importance
1    subdomain_count    0.334623
482          tld_top    0.052723
0             length    0.031399
239         tld_help   